# EDA - Data Cleaning (Multi-Ticker)

This notebook cleans all crypto tickers in the dataset and exports a combined, cleaned CSV file for clustering analysis.

In [6]:
import sys
import os
import pandas as pd
from dotenv import load_dotenv

# Setup paths to include src
sys.path.insert(0, os.path.abspath('src'))

from di.container import AppContainer

load_dotenv()
print("Environment loaded.")

Environment loaded.


## 1. Initialize Dependency Injection Container

In [7]:
container = AppContainer()
container.config.from_dict({
    "aws": {
        "s3_bucket_name": os.getenv("AWS_S3_BUCKET_NAME"),
        "region_name": os.getenv("AWS_REGION_NAME")
    },
    "db": {
        "connection_string": os.getenv("DB_CONNECTION_STRING")
    }
})

rds_to_pandas_usecase = container.rds_to_pandas_usecase()
data_cleaning_usecase = container.data_cleaning_usecase()
print("DI Container ready.")

DI Container ready.


## 2. Retrieve Data for All Tickers

In [ ]:
TICKERS_CONFIG = os.getenv("YAHOO_FINANCE_TICKERS", "BTC-USD,ETH-USD,SOL-USD,BNB-USD,DOGE-USD")
TICKERS = TICKERS_CONFIG.split(',')
TABLE_NAME = os.getenv("AWS_RDS_TABLE_NAME", "crypto_market_data")

all_dfs = []
for ticker in TICKERS:
    print(f"Fetching {ticker}...")
    df_ticker = rds_to_pandas_usecase.execute(ticker=ticker, table_name=TABLE_NAME)
    if not df_ticker.empty:
        all_dfs.append(df_ticker)

raw_df = pd.concat(all_dfs, ignore_index=True)
print(f"Retrieved {len(raw_df)} total records.")
raw_df.head()

Fetching BTC-USD...
2026-04-01 16:39:23,711 | INFO     | [App] Starting RDS -> Pandas retrieval for ticker: BTC-USD


## 3. Data Cleaning
Apply our `DataCleaningUseCase` (handles renames, missing values, anomalies).

In [ ]:
clean_df = data_cleaning_usecase.execute(raw_df)
print("Data cleaning complete.")
clean_df.head()

2026-04-01 16:37:16,244 | INFO     | [App] Starting data cleaning and preprocessing...
2026-04-01 16:37:16,246 | INFO     | [App] Dropped 'source' column.
2026-04-01 16:37:16,247 | INFO     | [App] Columns after renaming: ['id', 'ticker', 'date', 'open', 'high', 'low', 'close', 'volume']
2026-04-01 16:37:16,254 | INFO     | [App] Data cleaning complete. Missing: 0
Data cleaning complete.


,id,ticker,date,open,high,low,close,volume
0,1,BTC-USD,2024-01-01,42280.234375,44175.437500,42214.976562,44167.332031,18426978443
1,2,BTC-USD,2024-01-02,44187.140625,45899.707031,44176.949219,44957.968750,39335274536
2,3,BTC-USD,2024-01-03,44961.601562,45503.242188,40813.535156,42848.175781,46342323118
3,4,BTC-USD,2024-01-04,42855.816406,44770.023438,42675.175781,44179.921875,30448091210
4,5,BTC-USD,2024-01-05,44192.980469,44353.285156,42784.718750,44162.691406,32336029347


## 4. Export Cleaned Data for Clustering
We save this as a CSV file to be consumed by `eda_asset_clustering.ipynb`.

In [ ]:
OUTPUT_FILE = "cleaned_all_assets.csv"
clean_df.to_csv(OUTPUT_FILE, index=False)
print(f"All cleaned data saved to {OUTPUT_FILE}")

All cleaned data saved to cleaned_all_assets.csv
